# Session 5 demo, part B: the chi-square audit

Every test so far was about a mean. Chi-square is different: it works on **counts in a table**.
Here the question is an audit one: *are journal-entry errors spread evenly across departments, or
are they concentrated somewhere?*

It reproduces the numbers on the Part B slides and runs in Colab with nothing to upload.

## 1. The contingency table

An auditor sampled journal entries in four departments and flagged the suspect ones. The table
crosses **department** with **entry status** (flagged or clean).

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

depts = ["Sales", "Procurement", "Payroll", "Finance"]
table = pd.DataFrame(
    {"flagged": [45, 80, 15, 20], "clean": [455, 320, 285, 380]},
    index=depts,
)
table["flag_rate"] = (table["flagged"] / (table["flagged"] + table["clean"]) * 100).round(1)
print(table)

overall = table["flagged"].sum() / (table["flagged"].sum() + table["clean"].sum())
print(f"\noverall flag rate: {overall*100:.1f}%")

             flagged  clean  flag_rate
Sales             45    455        9.0
Procurement       80    320       20.0
Payroll           15    285        5.0
Finance           20    380        5.0

overall flag rate: 10.0%


**Predict first.** Look at the flag rates by department. If flags were *independent* of department,
every department would flag at about the overall rate. Do you think they do? Which department
stands out? Write it down before computing.

In [2]:
# Chi-square test of independence. One function returns the statistic, p-value, degrees of
# freedom, and the expected counts (what independence would predict).
observed = table[["flagged", "clean"]].values
chi2, p, dof, expected = stats.chi2_contingency(observed)

print(f"chi-square statistic = {chi2:.1f}")
print(f"degrees of freedom   = {dof}   (rows-1) x (cols-1)")
print(f"p-value              = {p:.2e}")

alpha = 0.05
print("\nReject independence: flags depend on department." if p < alpha
      else "\nNo evidence that flags depend on department.")

chi-square statistic = 64.4
degrees of freedom   = 3   (rows-1) x (cols-1)
p-value              = 6.59e-14

Reject independence: flags depend on department.


In [3]:
# Expected counts under independence, next to what was observed.
exp = pd.DataFrame(expected.round(1), index=depts, columns=["flagged_expected", "clean_expected"])
compare = pd.concat([table[["flagged", "clean"]], exp], axis=1)
print(compare)

             flagged  clean  flagged_expected  clean_expected
Sales             45    455              50.0           450.0
Procurement       80    320              40.0           360.0
Payroll           15    285              30.0           270.0
Finance           20    380              40.0           360.0


## 2. Which cells drive the result?

Chi-square adds up `(observed - expected)^2 / expected` over every cell. The cells with the biggest
gaps contribute most. Looking at the per-cell contributions tells the auditor *where* the
dependence lives.

In [4]:
contrib = (observed - expected) ** 2 / expected
contrib_df = pd.DataFrame(contrib.round(1), index=depts, columns=["flagged", "clean"])
contrib_df["total"] = contrib_df.sum(axis=1)
print(contrib_df.sort_values("total", ascending=False))
print(f"\nsum of all cells = {contrib.sum():.1f}  (this is the chi-square statistic)")

             flagged  clean  total
Procurement     40.0    4.4   44.4
Finance         10.0    1.1   11.1
Payroll          7.5    0.8    8.3
Sales            0.5    0.1    0.6

sum of all cells = 64.4  (this is the chi-square statistic)


## 3. From statistic to audit decision

The test does more than return a p-value. It tells the auditor where the risk is concentrated.

In [5]:
driver = contrib_df["total"].idxmax()
print(f"The dependence is driven by: {driver}")
print(f"{driver} flags at {table.loc[driver, 'flag_rate']:.0f}%, against an overall {overall*100:.0f}%.")
print("\nAudit response: put more sample and more senior review on", driver + ".")
print("This is risk-based audit sampling: the same test runs on controls by location,")
print("exceptions by system, or anomalies by vendor. It is a staple of audit analytics.")

The dependence is driven by: Procurement
Procurement flags at 20%, against an overall 10%.

Audit response: put more sample and more senior review on Procurement.
This is risk-based audit sampling: the same test runs on controls by location,
exceptions by system, or anomalies by vendor. It is a staple of audit analytics.


## 4. The same test on election night: exit polls

The same test of independence, on something everyone recognises. An exit poll crosses **vote
choice** (Candidate A or B) with a demographic such as **age group**. Illustrative and
non-partisan: hypothetical counts, generic candidates, no parties.

In [6]:
age_groups = ["18-29", "30-44", "45-64", "65+"]
exit_poll = pd.DataFrame(
    {"Candidate A": [320, 300, 250, 190], "Candidate B": [180, 240, 330, 400]},
    index=age_groups,
)
exit_poll["A_share"] = (exit_poll["Candidate A"] /
                        (exit_poll["Candidate A"] + exit_poll["Candidate B"]) * 100).round(1)
print(exit_poll)

e_obs = exit_poll[["Candidate A", "Candidate B"]].values
e_chi2, e_p, e_dof, e_exp = stats.chi2_contingency(e_obs)
print(f"\nchi-square = {e_chi2:.1f}, df = {e_dof}, p = {e_p:.2e}")
print("Reject independence: vote choice depends on age." if e_p < 0.05
      else "No evidence that vote choice depends on age.")

       Candidate A  Candidate B  A_share
18-29          320          180     64.0
30-44          300          240     55.6
45-64          250          330     43.1
65+            190          400     32.2

chi-square = 128.2, df = 3, p = 1.32e-27
Reject independence: vote choice depends on age.


**Same test, different table.** This is identical machinery to the audit example: state the null
(independence), run chi-square, read the p-value, decide. A pollster uses it to find the story in
the crosstab; an auditor uses it to steer the sample. One honest caveat: a real exit poll is
weighted and has sampling error, so a tiny p-value says the age split is real, it does not say the
headline number is exact.

---

### Ask Claude

> *I ran a chi-square test of independence on journal-entry flags by department: chi-square = 64,
> df = 3, p < 0.001. The Procurement department drives the result. Write two sentences I could put
> in an audit memo explaining what this means and what we will do about it, without overclaiming
> that Procurement is doing anything wrong.*

The last clause matters: a high flag rate is a reason to look, not a finding of fault.